# Data Cleaning Toolkit: step by step

This notebook runs the same pipeline as `clean.py`, one file at a time, so you can inspect each result. Use it to audit the cleaning, test config changes, or walk a client through the process.

In [ ]:
import pandas as pd
from cleaning.config import load_config
from cleaning.pipeline import clean_file
from cleaning.report import build_excel_report

pd.set_option('display.max_columns', None)

## 1. Load the config

In [ ]:
config = load_config('config_nobo.yaml')
config['columns']

## 2. Run the pipeline on one file (verbose = explains every step)

In [ ]:
result = clean_file('../NOBO-OP.csv', config, output_dir='output_notebook', verbose=True)
result['summary']

## 3. Look at the cleaned data

In [ ]:
df = result['data']
df[['shares', 'holder_name', 'entity_type', 'city', 'state_province', 'postal_code', 'country', 'is_international', 'parse_status']].head(20)

## 4. Rows sent to manual review, and why

In [ ]:
review = pd.read_csv(result['review_path'])
review['review_reason'].value_counts().head(15)

In [ ]:
review.head(20)

## 5. Unique investors

In [ ]:
result['extra']['investors'].head(20)

## 6. Test the address parser on a single row
Useful when a row is parsed wrongly: paste its lines here, check the result, then change `cleaning/address.py`.

In [ ]:
from cleaning.address import parse_lines
parse_lines(['JOHN SMITH', 'CHARLES SCHWAB & CO INC CUST', 'IRA ROLLOVER', 'TWO GREENWICH PLAZA', 'GREENWICH', 'CT 06830-'])

## 7. Excel quality report for this file

In [ ]:
x = result['extra']
build_excel_report([result['summary']], result['detail_log'], 'output_notebook/quality_report_notebook.xlsx',
                   {'reconciliation': [x['reconciliation']], 'investors': [x['investors']],
                    'collation_info': [x['collation_info']], 'eda': [x['eda']]})